# 03 — Two-Stage Default · Combine (M × N grid + Position Weighted Avg)

CLF prob × REG pred → 모든 (clf, reg) 조합 unit pred. 4 clf × 5 reg = **20 그리드 조합**.

- **Step 1**: die-level `final_die = clf_prob × reg_pred`
- **Step 2 (a)**: die→unit mean → `oof|val|test_unit.csv`
- **Step 2 (b)**: position weighted avg (Optuna 50 trial × 그리드) → `oof|val|test_unit_weighted.csv`
- **출력**: `4_output/03_two_stage/default/combined/{clf}_x_{reg}/...` + `grid_summary.csv` + `weighted_summary.csv` + `combine_meta.json`


## 1. 환경 + 모듈 자동 탐지

In [2]:
import os, sys, json

# Colab이면 setup만, 로컬이면 ../../../setup.py
try:
    import google.colab
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Two-Stage 산출물 위치: clf/<model>/<exp>/{oof,val,test}_die.csv (확률), reg/<model>/<exp>/{...}_die.csv (예측). combined/는 출력
# <exp>는 모델링 노트북의 EXP_ID 끝자리('001','002',...)로 만든 하위폴더. 모델마다 여러 실험이 쌓일 수 있어,
# 여기서는 모델별로 OOF 성능이 가장 좋은 실험 1개를 자동 선택한다 (clf=OOF AUC↑, reg=OOF RMSE↓).
GRID_ROOT = os.path.join(OUTPUT_DIR, '03_two_stage', 'default')
CLF_DIR   = os.path.join(GRID_ROOT, 'clf')
REG_DIR   = os.path.join(GRID_ROOT, 'reg')
OUT_DIR   = os.path.join(GRID_ROOT, 'combined')
os.makedirs(OUT_DIR, exist_ok=True)

N_TRIALS_POS = 1            # position 가중치 Optuna sub-study의 trial 수 (그리드 조합마다 한 study)
TIMEOUT_SEC  = None         # 초 단위, None=무제한

REQ_DIE = ['oof_die.csv', 'val_die.csv', 'test_die.csv']   # 이 3개가 다 있어야 그 실험을 후보로
EXP_PIN = {}   # 특정 실험 강제 고정 시: {'lgbm': '001', 'xgb': '002', ...}. 비우면 OOF 성능 best 자동 선택

def _oof_score(exp_dir, kind):
    # 후보 실험 폴더의 OOF 성능 (낮을수록 좋게 정규화) — reg: RMSE, clf: -AUC. oof_unit.csv 없거나 못 구하면 None
    p = os.path.join(exp_dir, 'oof_unit.csv')
    if not os.path.exists(p):
        return None
    df = pd.read_csv(p)
    if 'health' not in df.columns:
        return None
    if kind == 'clf':
        if 'prob' not in df.columns:
            return None
        d = df.dropna(subset=['prob', 'health'])
        try:
            return -float(roc_auc_score((d['health'].values > 0).astype(int), d['prob'].values))
        except Exception:
            return None
    if 'pred' not in df.columns:
        return None
    d = df.dropna(subset=['pred', 'health'])
    if len(d) == 0:
        return None
    return float(np.sqrt(np.mean((d['pred'].values - d['health'].values) ** 2)))

def _resolve_exp(model_dir, kind):
    # model_dir 바로 아래에 REQ_DIE 있으면 그대로(구 구조), 아니면 숫자 하위폴더 후보 중 OOF 성능 best (없으면 번호 큰 것)
    if all(os.path.exists(os.path.join(model_dir, f)) for f in REQ_DIE):
        return model_dir
    cands = []
    for d in sorted(os.listdir(model_dir)):
        sub = os.path.join(model_dir, d)
        if not (d.isdigit() and os.path.isdir(sub)):
            continue
        if not all(os.path.exists(os.path.join(sub, f)) for f in REQ_DIE):
            continue
        cands.append((d, sub, _oof_score(sub, kind)))
    if not cands:
        return None
    scored = [c for c in cands if c[2] is not None]
    if scored:
        return min(scored, key=lambda c: c[2])[1]
    return max(cands, key=lambda c: c[0])[1]

def _list_models(root, kind):
    # root 아래 각 모델 폴더에서 실험 1개씩 골라 {모델명: 실험폴더경로} dict로
    if not os.path.exists(root): return {}
    out = {}
    for name in sorted(os.listdir(root)):
        d = os.path.join(root, name)
        if not os.path.isdir(d): continue
        pin = EXP_PIN.get(name)
        path = os.path.join(d, pin) if pin else _resolve_exp(d, kind)
        if path and all(os.path.exists(os.path.join(path, f)) for f in REQ_DIE):
            out[name] = path
    return out

clf_pool = _list_models(CLF_DIR, 'clf')   # 가용 분류기들 (모델별 OOF AUC best 실험)
reg_pool = _list_models(REG_DIR, 'reg')   # 가용 회귀기들 (모델별 OOF RMSE best 실험)
print(f'CLF 가용 ({len(clf_pool)}): ' + ', '.join(f'{n}→{os.path.basename(p)}' for n, p in clf_pool.items()))
print(f'REG 가용 ({len(reg_pool)}): ' + ', '.join(f'{n}→{os.path.basename(p)}' for n, p in reg_pool.items()))
print(f'Grid 조합 수: {len(clf_pool) * len(reg_pool)}')   # 모든 (clf, reg) 쌍을 시험

if not clf_pool: raise RuntimeError(f'CLF 산출물 없음: {CLF_DIR}')
if not reg_pool: raise RuntimeError(f'REG 산출물 없음: {REG_DIR}')

setup 완료
CLF 가용 (4): catboost→001, et→002, lgbm→001, xgb→001
REG 가용 (5): catboost→001, enet→001, et→002, lgbm→002, xgb→002
Grid 조합 수: 20


## 2. CLF/REG die-level 로드 + 정합 검증

In [3]:
_, ys = load_all()
y_train = ys['train'].set_index(KEY_COL)[TARGET_COL]
y_val   = ys['validation'].set_index(KEY_COL)[TARGET_COL]
y_test  = ys['test'].set_index(KEY_COL)[TARGET_COL]

def _load_die(path, split, value_col):
    # die-level csv에서 [KEY, DIE_KEY, value]만 뽑고 value 컬럼명을 'v'로 통일
    df = pd.read_csv(os.path.join(path, f'{split}_die.csv'))
    return df[[KEY_COL, DIE_KEY_COL, value_col]].rename(columns={value_col: 'v'})

# clf는 'prob' 컬럼, reg는 'pred' 컬럼을 die-level로 로드 (split 3종 × 모델별)
clf_die = {n: {sp: _load_die(p, sp, 'prob') for sp in ['oof','val','test']} for n, p in clf_pool.items()}
reg_die = {n: {sp: _load_die(p, sp, 'pred') for sp in ['oof','val','test']} for n, p in reg_pool.items()}

# 곱셈이 die 단위로 element-wise라, 모든 csv의 (KEY, DIE_KEY) 행 순서가 정확히 같아야 함 → 검증
ref_die = {}
for sp in ['oof', 'val', 'test']:
    ref = next(iter(clf_die.values()))[sp][[KEY_COL, DIE_KEY_COL]]   # 첫 clf를 기준으로
    ref_die[sp] = ref
    for name, d in {**clf_die, **reg_die}.items():
        cur = d[sp][[KEY_COL, DIE_KEY_COL]]
        if len(cur) != len(ref):
            raise ValueError(f'{sp}/{name}: row 수 {len(cur)} != ref {len(ref)}')
        if not (cur.values == ref.values).all():
            raise ValueError(f'{sp}/{name}: (KEY, DIE_KEY) 순서 불일치')

print('[정합 OK] 모든 clf/reg die-level csv 동일 (KEY, DIE_KEY) 순서')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[정합 OK] 모든 clf/reg die-level csv 동일 (KEY, DIE_KEY) 순서


## 3. M × N grid 곱셈 + unit mean + RMSE

In [4]:
def _unit_mean(die_df, value_col='v'):
    # die 예측 → unit 예측 (평균)
    return die_df.groupby(KEY_COL, sort=False)[value_col].mean()

def _rmse(p, y):
    # 예측 Series를 정답 순서에 맞춰 정렬 후 RMSE
    p = p.loc[y.index]
    return float(np.sqrt(np.mean((p.values - y.values) ** 2)))

summary_rows = []
combined_unit = {}   # {(clf, reg): {'oof': Series, 'val': Series, 'test': Series}}

# 모든 (clf, reg) 쌍에 대해: die-level로 clf_prob × reg_pred → unit 평균 → 3 split RMSE
for c_name in clf_pool:
    for r_name in reg_pool:
        unit_preds = {}
        for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
            cdf = clf_die[c_name][sp]
            rdf = reg_die[r_name][sp]
            final_die = cdf['v'].values * rdf['v'].values   # P(Y>0) × E[Y|Y>0] (die 단위)
            tmp = pd.DataFrame({KEY_COL: cdf[KEY_COL].values, 'v': final_die})
            unit_preds[sp] = _unit_mean(tmp)

        combined_unit[(c_name, r_name)] = unit_preds
        summary_rows.append({
            'clf': c_name, 'reg': r_name,
            'oof':  _rmse(unit_preds['oof'],  y_train),
            'val':  _rmse(unit_preds['val'],  y_val),
            'test': _rmse(unit_preds['test'], y_test),
        })

summary = pd.DataFrame(summary_rows).sort_values('val').reset_index(drop=True)   # val 좋은 조합이 위로
print('=== M × N Grid 결과 (val 오름차순) ===')
print(summary.to_string(index=False, float_format='%.6f'))

best_row = summary.iloc[0]
print(f'\nGrid best: {best_row["clf"]} × {best_row["reg"]} → val={best_row["val"]:.6f} test={best_row["test"]:.6f}')

=== M × N Grid 결과 (val 오름차순) ===
     clf      reg      oof      val     test
    lgbm       et 0.008244 0.005711 0.008413
     xgb       et 0.008243 0.005712 0.008414
catboost       et 0.008247 0.005718 0.008418
      et       et 0.008247 0.005725 0.008424
    lgbm catboost 0.008284 0.005756 0.008456
    lgbm     enet 0.008276 0.005756 0.008449
     xgb catboost 0.008284 0.005758 0.008457
     xgb     enet 0.008276 0.005758 0.008450
catboost catboost 0.008286 0.005761 0.008460
catboost     enet 0.008278 0.005761 0.008452
      et catboost 0.008287 0.005764 0.008463
      et     enet 0.008277 0.005764 0.008456
    lgbm     lgbm 0.008287 0.005766 0.008464
     xgb     lgbm 0.008288 0.005768 0.008465
catboost     lgbm 0.008290 0.005771 0.008467
    lgbm      xgb 0.008287 0.005771 0.008467
     xgb      xgb 0.008287 0.005772 0.008468
      et     lgbm 0.008289 0.005773 0.008470
catboost      xgb 0.008290 0.005775 0.008470
      et      xgb 0.008289 0.005778 0.008473

Grid best: lgbm × et 

## 3.5 분류 threshold τ 탐색 (strategy_common.md §9 / §18 매트릭스 APPLY)

각 (clf × reg) 조합마다 **die-level prob에 임계값 τ를 적용**하여 `prob < τ`이면 final=0으로 강제.

- τ ∈ [0.00, 0.05, 0.10, ..., 0.50] 그리드 (총 11개)
- **train OOF에서 best τ 탐색 → val 적용 → val 개선 시 채택** (§19 원칙)
- 채택되면 `combined_unit[(c, r)]`을 τ-적용본으로 덮어씀 → 이후 save/position weighting 이 τ-적용 결과 사용
- 1차 분석 (Recall 0.011) 컨텍스트에서 분류 무력화 보정 목적

In [5]:
# 분류 threshold τ: die-level prob < τ 인 die는 final을 0으로 강제 (분류가 약할 때 작은 양수 누수를 막는 안전장치)
TAU_GRID = np.arange(0.00, 0.55, 0.05)   # 0.00 ~ 0.50 step 0.05 (11개; 0.00 = τ 미적용)

def _apply_tau(cdf, rdf, tau):
    # die final = clf_prob × reg_pred, 단 clf_prob < tau 면 0. → unit 평균으로 반환
    prob = cdf['v'].values
    pred = cdf['v'].values * rdf['v'].values
    pred = np.where(prob >= tau, pred, 0.0)
    tmp = pd.DataFrame({KEY_COL: cdf[KEY_COL].values, 'v': pred})
    return _unit_mean(tmp)

# 조합마다: train OOF에서 best τ를 찾고 → val에서 실제 개선되면 채택(combined_unit 덮어씀), 아니면 τ=0 유지
tau_decisions = []
for c_name in clf_pool:
    for r_name in reg_pool:
        cdf_oof = clf_die[c_name]['oof']
        rdf_oof = reg_die[r_name]['oof']
        baseline = combined_unit[(c_name, r_name)]            # τ=0 (위 cell의 결과)
        baseline_oof_rmse = _rmse(baseline['oof'], y_train)
        baseline_val_rmse = _rmse(baseline['val'], y_val)

        # 1) train OOF에서 RMSE가 가장 낮아지는 τ 탐색 (τ=0 baseline보다 나아야 함)
        best_tau, best_train_rmse = 0.0, baseline_oof_rmse
        for tau in TAU_GRID:
            if tau == 0.0:
                continue   # baseline과 동일
            cand = _apply_tau(cdf_oof, rdf_oof, tau)
            r = _rmse(cand, y_train)
            if r < best_train_rmse:
                best_train_rmse, best_tau = r, float(tau)

        if best_tau == 0.0:
            tau_decisions.append({'clf': c_name, 'reg': r_name,
                                  'best_tau': 0.0, 'accepted': False,
                                  'reason': 'train OOF에서 개선 없음'})
            continue

        # 2) 그 τ를 val/test에도 적용 → val이 실제로 좋아질 때만 채택
        cand_val_unit  = _apply_tau(clf_die[c_name]['val'],  reg_die[r_name]['val'],  best_tau)
        cand_test_unit = _apply_tau(clf_die[c_name]['test'], reg_die[r_name]['test'], best_tau)
        cand_train_unit = _apply_tau(cdf_oof, rdf_oof, best_tau)
        cand_val_rmse  = _rmse(cand_val_unit,  y_val)

        if cand_val_rmse + 1e-9 < baseline_val_rmse:
            # 채택 — 이후 save / position weighting이 τ-적용 결과를 쓰도록 combined_unit 덮어씀
            combined_unit[(c_name, r_name)] = {
                'oof':  cand_train_unit,
                'val':  cand_val_unit,
                'test': cand_test_unit,
            }
            tau_decisions.append({
                'clf': c_name, 'reg': r_name,
                'best_tau': best_tau, 'accepted': True,
                'baseline_val': baseline_val_rmse, 'tuned_val': cand_val_rmse,
                'delta_val': cand_val_rmse - baseline_val_rmse,
            })
        else:
            tau_decisions.append({
                'clf': c_name, 'reg': r_name,
                'best_tau': best_tau, 'accepted': False,
                'baseline_val': baseline_val_rmse, 'tuned_val': cand_val_rmse,
                'reason': 'val 개선 없음 → 미적용',
            })

tau_df = pd.DataFrame(tau_decisions)
print('=== τ 탐색 결과 ===')
print(tau_df.to_string(index=False, float_format='%.6f'))

# τ 적용 결과를 반영해 summary 다시 계산
summary_rows = []
for c_name in clf_pool:
    for r_name in reg_pool:
        u = combined_unit[(c_name, r_name)]
        summary_rows.append({
            'clf': c_name, 'reg': r_name,
            'oof':  _rmse(u['oof'],  y_train),
            'val':  _rmse(u['val'],  y_val),
            'test': _rmse(u['test'], y_test),
        })
summary = pd.DataFrame(summary_rows).sort_values('val').reset_index(drop=True)
print('\n=== τ 적용 후 M × N Grid (val 오름차순) ===')
print(summary.to_string(index=False, float_format='%.6f'))
best_row = summary.iloc[0]
print(f'\nGrid best: {best_row["clf"]} × {best_row["reg"]} → val={best_row["val"]:.6f} test={best_row["test"]:.6f}')

=== τ 탐색 결과 ===
     clf      reg  best_tau  accepted            reason  baseline_val  tuned_val  delta_val
catboost catboost  0.000000     False train OOF에서 개선 없음           NaN        NaN        NaN
catboost     enet  0.000000     False train OOF에서 개선 없음           NaN        NaN        NaN
catboost       et  0.150000      True               NaN      0.005718   0.005714  -0.000004
catboost     lgbm  0.000000     False train OOF에서 개선 없음           NaN        NaN        NaN
catboost      xgb  0.000000     False train OOF에서 개선 없음           NaN        NaN        NaN
      et catboost  0.150000      True               NaN      0.005764   0.005762  -0.000001
      et     enet  0.150000      True               NaN      0.005764   0.005762  -0.000001
      et       et  0.150000      True               NaN      0.005725   0.005721  -0.000004
      et     lgbm  0.150000      True               NaN      0.005773   0.005772  -0.000001
      et      xgb  0.000000     False train OOF에서 개선 없음         

## 4. 모든 조합 unit csv 저장 (mean baseline)

In [6]:
# 모든 (clf, reg) 조합 × 3 split의 unit 예측을 csv로 저장 (mean baseline, τ 채택된 조합은 τ-적용본)
for (c_name, r_name), preds in combined_unit.items():
    sub = os.path.join(OUT_DIR, f'{c_name}_x_{r_name}')
    os.makedirs(sub, exist_ok=True)
    for sp, y_true in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = preds[sp]
        df = pd.DataFrame({
            KEY_COL: s.index.values,
            'pred':  s.values,
            'health': y_true.reindex(s.index).values,   # 정답도 같이 (val/test는 비공개 라벨일 수 있음)
        })
        df.to_csv(os.path.join(sub, f'{sp}_unit.csv'), index=False)

summary.to_csv(os.path.join(OUT_DIR, 'grid_summary.csv'), index=False)
print(f'저장 완료: {OUT_DIR}')
print(f'  {len(combined_unit)}개 조합 × 3 split = {len(combined_unit)*3}개 csv')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\combined
  20개 조합 × 3 split = 60개 csv


## 5. Position weighted avg (그리드별 Optuna)

각 (clf, reg) 그리드 조합마다 die→unit 집계를 mean 대신 position 가중평균. Optuna가 4 weight (Dirichlet 정규화) 탐색.

In [7]:
# die→unit 집계를 mean 대신 position 가중평균으로 — 조합마다 가중치 w1~w4를 Optuna로 탐색
xs_full, _ = load_all()
pos_map = xs_full.set_index(DIE_KEY_COL)['position']   # DIE_KEY → position(1~4)

def _add_pos(ref_df):
    df = ref_df.copy()
    df['position'] = df[DIE_KEY_COL].map(pos_map).astype(int)
    return df

ref_oof_pos  = _add_pos(ref_die['oof'])
ref_val_pos  = _add_pos(ref_die['val'])
ref_test_pos = _add_pos(ref_die['test'])

def _pivot_die(uid, pos, vals):
    # die 값을 (unit × position) 행렬로 펼침
    df = pd.DataFrame({KEY_COL: uid, 'pos': pos, 'v': vals})
    return df.pivot(index=KEY_COL, columns='pos', values='v')

weighted_results = {}

print('=== Position weighted avg (그리드별 Optuna) ===\n')
for c_name in clf_pool:
    for r_name in reg_pool:
        # die-level final = clf_prob × reg_pred (τ는 여기선 안 씀 — 원본 곱으로 position 가중치를 봄)
        final_oof  = clf_die[c_name]['oof']['v'].values  * reg_die[r_name]['oof']['v'].values
        final_val  = clf_die[c_name]['val']['v'].values  * reg_die[r_name]['val']['v'].values
        final_test = clf_die[c_name]['test']['v'].values * reg_die[r_name]['test']['v'].values

        pivot_oof  = _pivot_die(ref_oof_pos[KEY_COL].values,  ref_oof_pos['position'].values,  final_oof)
        pivot_val  = _pivot_die(ref_val_pos[KEY_COL].values,  ref_val_pos['position'].values,  final_val)
        pivot_test = _pivot_die(ref_test_pos[KEY_COL].values, ref_test_pos['position'].values, final_test)

        # mean 집계 baseline RMSE (가중치를 균등 0.25로 본 것)
        baseline_oof  = float(np.sqrt(np.mean((pivot_oof.mean(axis=1).reindex(y_train.index).values  - y_train.values)**2)))
        baseline_val  = float(np.sqrt(np.mean((pivot_val.mean(axis=1).reindex(y_val.index).values    - y_val.values)**2)))
        baseline_test = float(np.sqrt(np.mean((pivot_test.mean(axis=1).reindex(y_test.index).values  - y_test.values)**2)))

        # 정답 순서에 맞춘 (unit × 4) 행렬 + 정답 벡터
        piv_oof_arr  = pivot_oof.reindex(y_train.index).values
        piv_val_arr  = pivot_val.reindex(y_val.index).values
        piv_test_arr = pivot_test.reindex(y_test.index).values
        y_tr_arr, y_vl_arr, y_te_arr = y_train.values, y_val.values, y_test.values

        # objective를 루프 안에서 매번 재정의 — 기본인자 _po/_yo로 이 조합의 OOF 행렬/정답을 바인딩 (closure 늦은 바인딩 함정 회피)
        def objective(trial, _po=piv_oof_arr, _yo=y_tr_arr):
            w_raw = np.array([trial.suggest_float(f'w{p}', 0.05, 1.0) for p in [1,2,3,4]])
            w = w_raw / w_raw.sum()                                     # Dirichlet 정규화 → 합=1
            return float(np.sqrt(np.mean(((_po * w).sum(axis=1) - _yo)**2)))   # train OOF 가중평균 RMSE

        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS_POS, timeout=TIMEOUT_SEC, show_progress_bar=False)

        bw = np.array([study.best_trial.params[f'w{p}'] for p in [1,2,3,4]])
        bw = bw / bw.sum()   # best 가중치 (정규화)

        # 그 가중치로 3 split 가중평균 예측
        w_oof  = (piv_oof_arr  * bw).sum(axis=1)
        w_val  = (piv_val_arr  * bw).sum(axis=1)
        w_test = (piv_test_arr * bw).sum(axis=1)

        weighted_results[(c_name, r_name)] = {
            'baseline':     {'oof': baseline_oof, 'val': baseline_val, 'test': baseline_test},
            'weighted':     {'oof': float(np.sqrt(np.mean((w_oof  - y_tr_arr)**2))),
                             'val': float(np.sqrt(np.mean((w_val  - y_vl_arr)**2))),
                             'test':float(np.sqrt(np.mean((w_test - y_te_arr)**2)))},
            'best_weights': bw.tolist(),
            'pred':         {'oof': pd.Series(w_oof,  index=y_train.index),
                             'val': pd.Series(w_val,  index=y_val.index),
                             'test':pd.Series(w_test, index=y_test.index)},
        }

        wo = weighted_results[(c_name, r_name)]['weighted']
        ba = weighted_results[(c_name, r_name)]['baseline']
        print(f'  [{c_name} × {r_name}] mean: oof={ba["oof"]:.6f} val={ba["val"]:.6f}  weighted: val={wo["val"]:.6f} (Δ={wo["val"]-ba["val"]:+.6f})')

# weighted summary 표 + csv
wr_rows = []
for (c, r), data in weighted_results.items():
    bw_ = data['best_weights']
    wr_rows.append({
        'clf': c, 'reg': r,
        'mean_oof':  data['baseline']['oof'],   'mean_val':  data['baseline']['val'],   'mean_test':  data['baseline']['test'],
        'weighted_oof': data['weighted']['oof'], 'weighted_val': data['weighted']['val'], 'weighted_test': data['weighted']['test'],
        'delta_val':  data['weighted']['val']  - data['baseline']['val'],
        'delta_test': data['weighted']['test'] - data['baseline']['test'],
        'w1': bw_[0], 'w2': bw_[1], 'w3': bw_[2], 'w4': bw_[3],
    })
weighted_summary = pd.DataFrame(wr_rows).sort_values('weighted_val').reset_index(drop=True)
weighted_summary.to_csv(os.path.join(OUT_DIR, 'weighted_summary.csv'), index=False)
print('\n=== Weighted summary (val 오름차순) ===')
print(weighted_summary.to_string(index=False, float_format='%.6f'))

# 가중평균 예측도 unit csv로 저장 (mean 버전과 별도 파일 *_unit_weighted.csv)
for (c, r), data in weighted_results.items():
    sub = os.path.join(OUT_DIR, f'{c}_x_{r}')
    os.makedirs(sub, exist_ok=True)
    for sp_key, y_ref in [('oof', y_train), ('val', y_val), ('test', y_test)]:
        s = data['pred'][sp_key]
        df = pd.DataFrame({
            KEY_COL: s.index.values,
            'pred':  s.values,
            'health': y_ref.reindex(s.index).values,
        })
        df.to_csv(os.path.join(sub, f'{sp_key}_unit_weighted.csv'), index=False)
print(f'\nweighted unit csv 저장 완료')

Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
=== Position weighted avg (그리드별 Optuna) ===

  [catboost × catboost] mean: oof=0.008286 val=0.005761  weighted: val=0.005762 (Δ=+0.000001)
  [catboost × enet] mean: oof=0.008278 val=0.005761  weighted: val=0.005762 (Δ=+0.000001)
  [catboost × et] mean: oof=0.008247 val=0.005718  weighted: val=0.005718 (Δ=+0.000000)
  [catboost × lgbm] mean: oof=0.008290 val=0.005771  weighted: val=0.005771 (Δ=+0.000000)
  [catboost × xgb] mean: oof=0.008290 val=0.005775  weighted: val=0.005775 (Δ=-0.000000)
  [et × catboost] mean: oof=0.008287 val=0.005764  weighted: val=0.005765 (Δ=+0.000002)
  [et × enet] mean: oof=0.008277 val=0.005764  weighted: val=0.005765 (Δ=+0.000001)
  [et × et] mean: oof=0.008247 val=0.005725  weighted: val=0.005725 (Δ=+0.000001)
  [et × lgbm] mean: oof=0.008289 val=0.005773  weighted: val=0.005774 (Δ=+0.000001)
  [et × xgb] mean: oof=0.008289 val=0.005778  weighted: val=0.005778 (Δ=+0.000000)
  [lgbm × catboost] 

## 6. 메타 저장

In [8]:
# combine 단계의 메타를 한 파일로 — pool 목록, grid best, weighted 결과 등
meta = {
    'clf_pool':  list(clf_pool),
    'reg_pool':  list(reg_pool),
    'n_grid_combinations': len(combined_unit),
    'grid_summary_top10': summary.head(10).to_dict(orient='records'),
    'grid_best_mean': {
        'clf': str(best_row['clf']), 'reg': str(best_row['reg']),
        'oof': float(best_row['oof']), 'val': float(best_row['val']), 'test': float(best_row['test']),
    },
    'weighted_grid': {
        'n_trials_per_study': N_TRIALS_POS,
        'timeout_sec':        TIMEOUT_SEC,
        'n_studies':          len(weighted_results),
        'summary_sorted':     weighted_summary.to_dict(orient='records'),
        'best_weighted_val':  float(weighted_summary.iloc[0]['weighted_val']),
        'best_combo':         f'{weighted_summary.iloc[0]["clf"]} × {weighted_summary.iloc[0]["reg"]}',
    },
}
with open(os.path.join(OUT_DIR, 'combine_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

# OUT_DIR 최상위의 파일들(요약 csv, meta)만 크기 출력 (조합별 하위 폴더는 제외)
for f_ in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f_)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {f_:30s}  {sz:>10,.1f} KB')

  combine_meta.json                     13.4 KB
  grid_summary.csv                       1.5 KB
  selected_versions.json                 0.6 KB
  weighted_summary.csv                   5.1 KB
